In [73]:
import numpy as np
import numpy.linalg as lin
import qutip as qt
import math

In [67]:
def sqrt(A):
    eigvals, eigvecs = lin.eig(A)
    eigvals = np.sqrt(eigvals)
    return np.matmul(np.matmul(eigvecs, np.diag(eigvals)), lin.inv(eigvecs))

In [ ]:
def tr_x(xyz, dx, dy, dz):
    return np.trace(xyz.reshape(dx, dy, dz, dx, dy, dz), axis1=0, axis2=3)

def tr_y(xyz, dx, dy, dz):
    return np.trace(xyz.reshape(dx, dy, dz, dx, dy, dz), axis1=1, axis2=4)

def tr_z(xyz, dx, dy, dz):
    return np.trace(xyz.reshape(dx, dy, dz, dx, dy, dz), axis1=2, axis2=5)

def tr_yz(xyz, dx, dy, dz):
    xy = tr_z(xyz, dx, dy, dz).reshape(dx*dy, dx*dy)
    return np.trace(xy.reshape(dx, dy, dx, dy), axis1=1, axis2=3)

def tr_xz(xyz, dx, dy, dz):
    xy = tr_z(xyz, dx, dy, dz).reshape(dx*dy, dx*dy)
    return np.trace(xy.reshape(dx, dy, dx, dy), axis1=0, axis2=2)

def tr_xy(xyz, dx, dy, dz):
    xz = tr_y(xyz, dx, dy, dz)
    return np.trace(xz.reshape(dx, dz, dx, dz), axis=0, axis = 2)

In [ ]:
def get_zlx(xyz, dx, dy, dz):
    xz = tr_y(xyz, dx, dy, dz)
    proj = np.kron(lin.inv(sqrt(tr_yz(xyz, dx, dy, dz))), np.eye(dz))
    return np.matmul(np.matmul(proj, xz), proj)

def get_zly(xyz, dx, dy, dz):
    yz = tr_x(xyz, dx, dy, dz)
    proj = np.kron(lin.inv(sqrt(tr_xz(xyz, dx, dy, dz))), np.eye(dz))
    return np.matmul(np.matmul(proj, yz), proj)
    

In [116]:
# A = np.array([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]]).reshape(4, 4)

# print(A)
#print(A.reshape(2, 2, 2, 2))
#print(np.trace(A.reshape(2,2,2,2), axis1=1, axis2=3))
# print(np.trace(A.reshape(2,2,2,2), axis1=0, axis2=1))

psi = qt.tensor(qt.basis(2, 0), qt.basis(2, 1), (qt.basis(2, 0) + qt.basis(2, 1)).unit())
x = np.kron(np.kron(np.array([[1, 0], [0, 0]]), np.array([[0, 0], [0, 1]])), np.array([[0.5, 0.5], [0.5, 0.5]]))

print(psi.ptrace([0]))
print(tr_yz(x, 2, 2, 2))

Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=True
Qobj data =
[[1. 0.]
 [0. 0.]]
[[1. 0.]
 [0. 0.]]


In [ ]:
def initC(xy_size, dz):
    A = []

In [ ]:
def QLatentSearch(esti_state, dx, dy, dz, smoothing, damping, log_reg, penalty, n):
    xy_size = esti_state.shape[0]
    smooth_esti = ((1-smoothing) * esti_state) + ((smoothing/xy_size) * np.eye(xy_size))
    C = initC(xy_size, dz)
    for i in range(n):
        condit_projector = np.kron(sqrt(smooth_esti), np.eye(dz))
        p_xyz = np.matmul(np.matmul(condit_projector, C), condit_projector)
        C_zlx = get_zlx(p_xyz, dx, dy, dz)
        C_zly = get_zly(p_xyz, dx, dy, dz)
        p_z = tr_xy(p_xyz, dx, dy, dz)

        